# 1. Load the NACC Dataset

In [ ]:
#import some needed library
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import MinMaxScaler

from matplotlib import pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
#import and load dataset
nacc = pd.read_csv('investigator_nacc71.csv')
nacc.head(10)

In [ ]:
#get basic info of dataset
nacc.shape

In [ ]:
#get basic info of dataset
nacc.info()

# 2. Basic Data Preprocessing

## a. Visit Selection

In [ ]:
#filter the dataset only keep the baselind visit for each subject
nacc_baseline = nacc[nacc['NACCVNUM'] == 1].copy()
print(f"Shape after Visit Selection:", nacc_baseline.shape)

## b. Manual Filtering

In [ ]:
#remove ID, visit, date, copartcipant data, milestone, neuropathology features & others
#*REFER to uds3-rdd.pdf
coltodrop = ['NACCID', 'NACCAGEB', 'NACCADC', 'PACKET', 'FORMVER', 'NACCVNUM', 'VISITMO', 'VISITYR', #id, form header, visit dates
             'VISITDAY','NACCAVST', 'NACCNVST', 'NACCDAYS', 'NACCFDYS', 'NACCREAS','NACCREFR', #form header, visit dates
             'BIRTHMO', 'BIRTHYR', 'INBIRMO', 'INBIRYR', 'INSEX', 'INHISP', 'INHISPOR', 'INRACE', 'INRASEC', #coparticipant data
             'INRATER', 'INEDUC', 'INRELTO', 'INKNOWN', 'INLIVWTH', 'INVISITS', 'INCALLS', 'INRELY', 'NACCNINR',  #coparticipant data
             'NACCACTV', 'NACCNOVS', 'NACCDSMO', 'NACCDSDY', 'NACCDSYR', 'NACCNURP', 'NACCNRMO', #milestones
             'NACCNRDY', 'NACCNRYR', 'NACCMDSS', 'NACCPAFF', 'NACCLIVS', 'NORMEXAM', 'INDEPEND', 'HISPANIC'] 

#remove neuropathology colomun by searching variable name start with NP
neurodrop = [c for c in nacc_baseline.columns if c.startswith('NP')]
#remove the rest neuropathology variables
#*REFER to rdd-np.pdf
neurodrop2 = ['NACCADC', 'NACCDAGE', 'NACCMOD', 'NACCDIED', 'NACCYOD', 'NACCINT',
              'NACCBRNN', 'NACCAVAS', 'NACCBRAA', 'NACCNEUR', 'NACCDIFF',
              'NACCVASC', 'NACCAMY', 'NACCINF', 'NACCMICR', 'NACCHEM',
              'NACCARTE', 'NACCNEC', 'NACCLEWY', 'NACCPICK', 'NACCCBD',
              'NACCPROG', 'NACCPRIO', 'NACCDOWN', 'NACCOTHP', 'NACCWRI1',
              'NACCWRI2', 'NACCWRI3', 'NACCBNKF', 'NACCFORM', 'NACCPARA',
              'NACCCSFP']

#post-diagnosis variables that not needed
target_leakage_var = ['DEMENTED', 'NACCNORM', 'NACCMCII', 'NACCIDEM', 'NACCNREX', 
                      'COGSTAT', 'NORMCOG', 'COGOTH', 'APA', 'OTHCOG', 'NACCNMRI',
                      'COGJUDG', 'AGIT', 'CVD', 'IRR', 'NACCAPOE', 'DXMETHOD', 'NACCMRSA']

clinical_judge_var = ['B9CHG', 'DECSUB', 'DECIN', 'DECCLIN', 'DECCLCOG', 'COGMEM',
                      'COGORI', 'COGLANG', 'COGVIS', 'COGATTN', 'COGFLUC', 'COGFLAGO',
                      'COGOTHR', 'COGOTHRX', 'NACCCOGF', 'NACCCGFX', 'COGMODE', 'COGMODEX',
                      'DECAGE', 'DECCLBE', 'BEAPATHY', 'BEDEP', 'BEVHALL', 'BEVWELL', 'BEVHAGO',
                      'BEAHALL', 'BEDEL', 'BEDISIN', 'BEIRRIT', 'BEAGIT', 'BEPERCH', 'BEREM',
                      'BEREMAGO', 'BEANX', 'BEOTHR', 'BEOTHRX', 'NACCBEHF', 'NACCBEFX', 
                      'BEMODE', 'BEMODEX', 'BEAGE', 'DECCLMOT', 'MOGAIT', 'MOFALLS',
                      'NACCMOTF', 'MOMODE', 'MOMODEX', 'MOMOPARK', 'PARKAGE', 'MOMOALS',
                      'ALSAGE', 'MOAGE', 'COURSE', 'FRSTCHG', 'LBDEVAL', 'FTLDEVAL']

#medical record crash with clinical diagnosis or not clinical that affect
med_his = ['HYPERTEN', 'ARTHRIT', 'ARTH', 'ARTHSPIN', 'ARTHUPEX', 'ARTHLOEX', 'HYPERCHO', 
           'DIABETES', 'BPSYS', 'BPDIAS', 'DIABTYPE', 'HRATE', 'TOBAC100', 'HYPCHOL', 'DEP2YRS',
           'AFIBRILL', 'CONGHRT', 'STROKE', 'DEP', 'HEIGHT', 'WEIGHT', 'ALCOCCAS', 'TOBAC30',
           'OTHCOND', 'OTHCONDX', 'OTHNEUR', 'OTHNEURX', 'APNEA', 'SLEEPAP', 'HEARING', 'HEARAID',
           'HEARWAID', 'VISCORR', 'VISWCORR', 'DEPOTHR', 'DEPIF', 'DEPTREAT', 'THYROID', 'VISION',
           'DEPOTHR', 'CANCER', 'THYDIS', 'URINEINC', 'INCONTU', 'HANDED', 'HYPOSOM', 'NACCMOM', 
           'NACCFAM', 'NACCDAD', 'INSOMN', 'TBI', 'CVOTHR', 'B12DEF', 'PACKSPER']

#other genetic variables
gen_var = ['ADGCGWAS', 'ADGCEXOM', 'ADGCRND', 'ADGCEXR', 'NGDSGWAS', 'NGDSEXOM',
           'NGDSWGS', 'NGDSWES', 'NGDSGWAC', 'NGDSEXAC', 'NGDSWGAC', 'NACCNCRD']

#medication variable 
med_var = ['NACCAMD', 'NACCAC', 'NACCAPSY', 'NACCADEP', 'NACCAHTN', 'NACCHTNC',
           'NACCACEI', 'NACCAAAS', 'NACCBETA', 'NACCCCBS', 'NACCDIUR', 'NACCVASD',
           'NACCANGI', 'NACCLIPL', 'NACCNSD', 'NACCAC', 'NACCADEP', 'NACCAPSY',
           'NACCAANX', 'NACCADMD', 'NACCPDMD', 'NACCEMD', 'NACCEPMD', 'NACCDBMD']

coltodrop.extend(neurodrop)
coltodrop.extend(neurodrop2)
coltodrop.extend(target_leakage_var)
coltodrop.extend(clinical_judge_var)
coltodrop.extend(med_his)
coltodrop.extend(gen_var)
coltodrop.extend(med_var)


nacc_new = nacc_baseline.drop(columns=coltodrop)
print(f"Total columns to drop: {len(coltodrop)}")
print(f"Shape after Manual Filtering:", nacc_new.shape)

In [ ]:
# Drop MMSE Features (Keep MoCA) *MoCA is sensitive than MMSE
mmse_cols = [c for c in nacc_new.columns if 'MMSE' in c]
mmse_cols.extend(['PENTAGON'])

#Drop MoCA subvariables to keep only the Total Score, NACCMOCA
moca_subs = [c for c in nacc_new.columns if c.startswith('MOCA')]

#Drop CDR subvarible to keep only total, CDRSUM
cdr_subs = ['MEMORY', 'ORIENT', 'JUDGMENT', 'COMMUN', 'HOMEHOBB', 'PERSCARE', 
            'CDRGLOB', 'COMPORT', 'CDRLANG']

#drop gds sub score to keep total only, NACCGDS
gds_subs = ['NOGDS', 'SATIS', 'DROPACT', 'EMPTY', 'BORED', 'SPIRITS', 'AFRAID',
            'HAPPY', 'HELPLESS', 'STAYHOME', 'MEMPROB', 'WONDRFUL', 'WRTHLESS',
            'ENERGY', 'HOPELESS', 'BETTER']

#too many Neuropsychological tests taking more time and some tests test the same function 
#so only keep important test (Trail A and B) that are not overlap 
other_test = ['TRAILARR', 'TRAILALI', 'TRAILBRR', 'TRAILBLI', 'CRAFTDRE', 'CRAFTDVR',
             'ANIMALS', 'VEG', 'CRAFTVRS', 'CRAFTURS', 'UDSBENTC', 'UDSBENRS', 'UDSBENTD',
             'UDSVERFC', 'UDSVERFN', 'UDSVERNF', 'UDSVERLC', 'UDSVERLR', 'UDSVERLN',
             'UDSVERTN', 'UDSVERTE', 'UDSVERTI', 'DIGFORCT', 'DIGFORSL', 'DIGBACCT',
             'DIGIF', 'DIGIFLEN', 'DIGIB', 'DIGIBLEN', 'DIGBACLS', 'MINTTOTW', 'MINTSCNG',
             'MINTSCNC', 'MINTPCNG', 'MINTPCNC', 'CRAFTCUE', 'CRAFTDTI']

#other diagnosis variables  
other_var = ['DYSILL', 'FTLDNOS', 'BRNINJ', 'PARKSIGN', 'MOSLOW',  
             'STOVE', 'DIGFORSL', 'DIGBACLS', 'DIGBACCT']
#npiq variable
npiq_vars = ['DEL', 'HALL', 'AGIT', 'DEPD', 'ANX', 'ELAT', 'APA', 'DISN', 'IRR', 
             'MOT', 'NITE', 'APP']

total_drop = mmse_cols + moca_subs + cdr_subs + gds_subs + other_test + other_var
nacc_new2 = nacc_new.drop(columns=total_drop)
nacc_new2 = nacc_new2.drop(columns=[c for c in nacc_new.columns if any(p in c for p in npiq_vars)], errors='ignore')

print(f"New Shape after filtering: {nacc_new2.shape}")

In [ ]:
#Regroup the target variable NACCUDSD
#Drop code 2 Impaired-Not-MCI cuz it is inconsistently defined and heterogeneous
# not quite normal and not quite MCI so remove it to make it cleaner
nacc_new3 = nacc_new2[nacc_new2['NACCUDSD'] != 2].copy()

#filter dementia, only keep Alzheimer's Disease Dementia (project focus)
#use NACCALZD to keep AD dementia from NACCUDSD
#keep 3 classes including AD dementia
three_classes = (nacc_new3['NACCUDSD'].isin([1, 3])) | ((nacc_new3['NACCUDSD'] == 4) & (nacc_new3['NACCALZD'] == 1))
nacc_new3 = nacc_new3[three_classes].copy()

#Drop the NACCALZD column
nacc_new4 = nacc_new3.drop(columns=['NACCALZD'])

print(f"New Shape: {nacc_new4.shape}")
print(f"Class Counts:\n{nacc_new4['NACCUDSD'].value_counts().sort_index()}")

## c. Data Cleaning

In [ ]:
# In NACC, missing values are shown as -4, replace with nan
nacc_clean = nacc_new3.replace(-4, np.nan)

#there are more codes mean unknown, not applicable like 8, 9, 88, 99, 888, 999, 8888, 9999
#replace with nan
trash_code_b = [88, 99, 888, 999, 8888, 9999, 88.8, 888.8]
trash_code_s = [8, 9]

#trail a and b have problem code 995-998, change to nan
if 'TRAILB' in nacc_clean.columns:
    nacc_clean['TRAILB'] = nacc_clean['TRAILB'].replace([995, 996, 997, 998], np.nan)

if 'TRAILA' in nacc_clean.columns:
    nacc_clean['TRAILA'] = nacc_clean['TRAILA'].replace([995, 996, 997, 998], np.nan)

#mint has 95-98 problem code, change to nan
if 'MINTTOTS' in nacc_clean.columns:
    nacc_clean['MINTTOTS'] = nacc_clean['MINTTOTS'].replace([95, 96, 97, 98], np.nan)


#variables that need careful clean because score range overlap with missing code
critical_var = ['NACCMOCA', 'NACCGDS', 'CDRSUM', 'EDUC', 'NACCUDSD', 'SMOKYRS']
critical_exist = [c for c in critical_var if c in nacc_clean.columns]
trail_var = ['TRAILA', 'TRAILB']
problem_s_var = ['MINTTOTS']

#other category col (category feature normally dont have more than 10 categories)
other_cols = set(trail_var + problem_s_var + critical_exist + ['NACCAGE'])
general_cols = [c for c in nacc_clean.columns if c not in other_cols]
cat_cols = [col for col in general_cols 
            if nacc_clean[col].nunique(dropna=True) <= 10]


#clean
nacc_clean[general_cols] = nacc_clean[general_cols].replace(trash_code_b, np.nan)
nacc_clean[critical_exist] = nacc_clean[critical_exist].replace(trash_code_b, np.nan)
nacc_clean[cat_cols] = nacc_clean[cat_cols].replace(trash_code_s, np.nan)

#check total missing values
print(f"Total Missing Values: {nacc_clean.isnull().sum().sum()}")

In [ ]:
#MoCA has many missing parts because it was only adopted since 2015
#check whether important variable NACCMoCA has how many percentage missing
#if more than 50% missing, cannot do imputation, remove missing rows instead
total_subject = len(nacc_clean)
moca_missing = nacc_clean['NACCMOCA'].isna().sum()
moca_missingpercent = (moca_missing / total_subject) * 100
print(f"MoCA Missing Percentage: {moca_missingpercent:.2f}%")

In [ ]:
#since important variable NACCMoCA has more than 50% missing, cannot impute
#only can remove missing rows
nacc_clean1 = nacc_clean.copy()
nacc_clean1 = nacc_clean1.dropna(subset=['NACCMOCA'])
miss_percent = nacc_clean1.isnull().sum() / len(nacc_clean1) * 100
cols_more50miss = miss_percent[miss_percent > 50].index
nacc_clean2 = nacc_clean1.drop(columns=cols_more50miss)

print(f"Final Subject: {nacc_clean2.shape[0]}")
print(f"Number of Features Left: {nacc_clean2.shape[1]}")
print(f"Total Missing Values Left: {nacc_clean2.isnull().sum().sum()}")

In [ ]:
nacc_clean2.info()

In [ ]:
#there are object type variables
#check the rest object whether it is important or not
object_cols = nacc_clean2.select_dtypes(include=['object']).columns.tolist()
print(f"The object columns are: {object_cols}")

In [ ]:
#clean all obj variables (not clinical predictors)
nacc_cleaned = nacc_clean2.drop(columns=object_cols)

print(f"New Shape: {nacc_cleaned.shape}")

In [ ]:
#Imputation for nan values
con_var = ['NACCAGE', 'EDUC', 'NACCMOCA', 'CDRSUM', 'NACCGDS', 'MINTTOTS',
            'TRAILA', 'TRAILB', 'SMOKYRS'] #influential continuous var
cat_var = [c for c in nacc_cleaned.columns if c not in con_var] #categorical var
#continuous nan replaced by median, categorical nan replaced by mode
nacc_cleaned[con_var] = nacc_cleaned[con_var].fillna(nacc_cleaned[con_var].median())
nacc_cleaned[cat_var] = nacc_cleaned[cat_var].fillna(nacc_cleaned[cat_var].mode().iloc[0])

print("~ Final Cleaning Results ~")
print(f"Total missing values: {nacc_cleaned.isnull().sum().sum()}") #must 0
print(f"Final Shape: {nacc_cleaned.shape}")

## d. Encoding

In [ ]:
#For target variable NACCUDSD, rename to target (easy to read)
# Mapping: 1 to 0 (Normal), 3 to 1 (MCI), 4 to 2 (AD Dementia)
target_map = {1: 0, 3: 1, 4: 2}
nacc_cleaned['Target'] = nacc_cleaned['NACCUDSD'].map(target_map)
#drop original NACCUDSD column
nacc_cleaned = nacc_cleaned.drop(columns=['NACCUDSD'])

print(f"Target Distribution:\n{nacc_cleaned['Target'].value_counts().sort_index()}")

In [ ]:
#for categorical variables: one-hot encoding except continuous data
nacc_encode = nacc_cleaned.copy()
continuous_var = ['NACCAGE', 'EDUC', 'NACCBMI', 'NACCMOCA', 'CDRSUM', 'NACCGDS', 
                  'MINTTOTS', 'TRAILA', 'TRAILB', 'SMOKYRS']
continuous_var = [c for c in continuous_var if c in nacc_encode.columns]
encode_col = [c for c in nacc_encode.columns if c not in continuous_var + ['Target']]
#encode 
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoded_array = ohe.fit_transform(nacc_encode[encode_col])
encoded_feature_names = ohe.get_feature_names_out(encode_col)
#create a DataFrame with encoded variables
encoded_df = pd.DataFrame(encoded_array, columns=encoded_feature_names, index=nacc_encode.index)
nacc_ml = pd.concat([nacc_encode[continuous_var], encoded_df, nacc_encode[['Target']]], axis=1)

print(f"Features after encoding: {nacc_ml.shape[1]}")

# Exploratory Data Analysis (EDA)

## a. Descriptive Statistics

In [ ]:
#Check distribution of target variable NACCUDSD
count = nacc_ml['Target'].value_counts()
percentage = nacc_ml['Target'].value_counts(normalize=True) * 100

#make a table
distribution = pd.DataFrame({'Count': count, 'Percentage': percentage.map('{:.4f}%'.format)})
distribution = distribution.sort_index() #the index is not arranged, need sort

#map index to names
names = {0: 'Normal Cognition', 1: 'MCI', 2: 'AD Dementia'}
distribution.index = distribution.index.map(names)

print("~ Target Variable Distribution ~")
display(distribution)

In [ ]:
#select cleaned numerical variables to describe because too many variables
#so just describe the important one
spec_cols = ['NACCAGE', 'EDUC', 'NACCMOCA', 'CDRSUM', 'NACCGDS', 'MINTTOTS',
            'TRAILA', 'TRAILB', 'SMOKYRS']
#then filter list to only columns that exist
existing_spec_cols = [c for c in spec_cols if c in nacc_ml.columns]
display(nacc_ml[existing_spec_cols].describe().round(4))

In [ ]:
#for detail univariate analysis
#this function is from data mining subject lab2, add Skewness, Kurtosis, Variance, Std Dev
def Univariate(dataset, quan):
    # Create a DataFrame with the specified index and columns
    descriptive = pd.DataFrame(index=[
        "Mean", "Median", "Mode", "Q1:25%", "Q2:50%", "Q3:75%", "Q4:100%",
        "IQR", "1.5rule", "Lesser", "Greater", "Min", "Max", "Skewness", "Kurtosis",
        "Variance", "Std Dev"
    ], columns=quan)

    # Loop through each quantitative column and calculate the statistics
    for columnName in quan:
        descriptive.loc["Mean", columnName] = round(nacc_ml[columnName].mean(), 1)
        descriptive.loc["Median", columnName] = round(nacc_ml[columnName].median(), 1)
        descriptive.loc["Mode", columnName] = round(nacc_ml[columnName].mode()[0], 1)
        descriptive.loc["Q1:25%", columnName] = round(nacc_ml[columnName].quantile(0.25), 1)
        descriptive.loc["Q2:50%", columnName] = round(nacc_ml[columnName].quantile(0.50), 1)  # same as median
        descriptive.loc["Q3:75%", columnName] = round(nacc_ml[columnName].quantile(0.75), 1)
        descriptive.loc["Q4:100%", columnName] = round(nacc_ml[columnName].max(), 1)
        descriptive.loc["IQR", columnName] = round(descriptive.loc["Q3:75%", columnName] - descriptive.loc["Q1:25%", columnName], 1)
        descriptive.loc["1.5rule", columnName] = round(1.5 * descriptive.loc["IQR", columnName], 1)
        descriptive.loc["Lesser", columnName] = round(descriptive.loc["Q1:25%", columnName] - descriptive.loc["1.5rule", columnName], 1)
        descriptive.loc["Greater", columnName] = round(descriptive.loc["Q3:75%", columnName] + descriptive.loc["1.5rule", columnName], 1)
        descriptive.loc["Min", columnName] = round(nacc_ml[columnName].min(), 1)
        descriptive.loc["Max", columnName] = round(nacc_ml[columnName].max(), 1)
        descriptive.loc["Skewness", columnName] = round(nacc_ml[columnName].skew(), 1)  # Add Skewness
        descriptive.loc["Kurtosis", columnName] = round(nacc_ml[columnName].kurt(), 1)  # Add Kurtosis
        descriptive.loc["Variance", columnName] = round(nacc_ml[columnName].var(), 1)   # Add Variance
        descriptive.loc["Std Dev", columnName] = round(nacc_ml[columnName].std(), 1)    # Add Std Dev

    return descriptive

descriptive_stat = Univariate(nacc_ml, spec_cols)
display(descriptive_stat)

## b. Visualisation

In [ ]:
#Target Class Distribution
targetnames = ['Normal', 'MCI', 'AD Dementia']
codes = [0,1,2]
barchart = sns.countplot(x='Target', data=nacc_ml, order=codes, palette='viridis')
#replace x axis number to names
plt.xticks(ticks=[0, 1, 2], labels=targetnames)

#add data label and make sure data label match the bar height
for p in barchart.patches:
    barchart.annotate(format(p.get_height(), '.0f'),
                   (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha = 'center', va = 'bottom')

plt.title('Class Distribution of Cognitive Status')
plt.xlabel('Cognitive Status')
plt.ylabel('Count')
plt.show()

In [ ]:
#Sex distribution in Pie chart
sexcount = nacc_ml['SEX_2'].value_counts()
labels = ['Female', 'Male'] 
sizes = [sexcount[1], sexcount[0]]

color = ['lightcoral', 'lightblue']

plt.pie(sizes, labels=labels, colors=color, autopct='%1.2f%%', startangle=90)
plt.axis('equal')
plt.title('Gender Distribution')
plt.show()

In [ ]:
#Age Histogram
sns.histplot(nacc_ml['NACCAGE'], bins=40, color='darkslateblue', edgecolor='black')
plt.title('Age Distribution of Subjects')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#MoCA score distribution
sns.histplot(nacc_ml['NACCMOCA'], bins=30, color='teal', edgecolor='black')
plt.title('MoCA Score Distribution')
plt.xlabel('MoCA Score (0-30)')
plt.ylabel('Frequency')
#add a vertical line for the "Normal" score (usually 26)
plt.axvline(26, color='orange', linestyle='--', label='Typical Normal Cutoff (26)')
plt.legend()
plt.show()

In [ ]:
#education histogram
sns.histplot(nacc_ml['EDUC'], bins=30, color='seagreen', edgecolor='black')
plt.title('Education Level Distribution')
plt.xlabel('Years of Education')
plt.ylabel('Frequency')
#put line show high school graduated or university graduated
plt.axvline(12, color='red', linestyle='--', label='High School (12 yrs)')
plt.axvline(16, color='blue', linestyle='--', label='Bachelor Degree (16 yrs)')
plt.legend()

plt.show()

In [ ]:
#sex vs diagnosis
sexbar = sns.countplot(x='Target', hue='SEX_2', data=nacc_ml, palette='coolwarm')

#add data label
for p in sexbar.patches:
  if p.get_height() > 0:
    sexbar.annotate(format(p.get_height(), '.0f'),
                   (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha = 'center', va = 'bottom')

plt.title('Cognitive Status Count Grouped by Gender')
plt.xlabel('Cognitive Status')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1, 2], labels=['Normal', 'MCI', 'AD Dementia'])
plt.legend(title='Gender', labels=['Male', 'Female'])
plt.show()

In [ ]:
sns.boxplot(x='Target', y='NACCMOCA', data=nacc_ml, palette='tab10')
plt.xticks(ticks=[0, 1, 2], labels=['Normal', 'MCI', 'AD Dementia'])
plt.title('MoCA Score Distribution by Cognitive Status')
plt.xlabel('Cognitive Status')
plt.ylabel('MoCA Score')
plt.show()

# 3. Feature Engineering

## a. Feature Selection

### Step 1: Data Splitting (80:20)

In [ ]:
#split first so the Test Set remains a "Real World" benchmark
X = nacc_ml.drop(columns=['Target'])
y = nacc_ml['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, 
                                                    random_state=42, stratify=y)

X_train.shape, X_test.shape

### Step 2: Drop Low-Variance Features

In [ ]:
#remove features that have 99% same values 
p = 0.99
selector = VarianceThreshold(threshold=(p* (1 - p)))
X_train_reduced = selector.fit_transform(X_train)
X_test_reduced = selector.transform(X_test)

#get the names of the remaining features
selected_cols = X_train.columns[selector.get_support()]

print(f"Features remaining after Variance Threshold: {len(selected_cols)}")

### Step 3: Correlation

In [ ]:
#Calculate the correlation matrix
#abs() used to catch both strong +ve and strong -ve correlation
X_before_corr = pd.DataFrame(X_train_reduced, columns=selected_cols)
corr_matrix = X_before_corr.corr().abs()

#select upper triangl (triu()) of correlation matrix (to avoid checking pairs twice e.g. AvsB & BvsA)
upper = corr_matrix.where(np.full(corr_matrix.shape, True, dtype=bool) &
                          np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

#get features with correlation more than 0.90 (reduce multicollinearity)
todrop = [column for column in upper.columns if any(upper[column] > 0.90)]

X_train_final = X_before_corr.drop(columns=todrop)
X_test_final = pd.DataFrame(X_test_reduced, columns=selected_cols).drop(columns=todrop)

print(f"Features dropped due to redundancy (>0.90): {len(todrop)}")
print(f"Features remaining after Correlation: {X_train_final.shape[1]}")
print(X_train_final.shape, X_test_final.shape)

### Step 4: Random Forest Feature Importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_final, y_train)

#take raw importance 
raw_importance = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Importance': rf.feature_importances_
})

#the variable are encoded, so need to group to see the top 30
#Extract the Variable Name (e.g. BILLS_1.0 to BILLS) to see the top 30
raw_importance['Base_var'] = raw_importance['Feature'].apply(lambda x: x.split('_')[0])

#Sum up the importance scores for each base variable
grouped_importance = raw_importance.groupby('Base_var')['Importance'].sum().sort_values(ascending=False).reset_index()

#Visualise Top 30 -_-
top_30 = grouped_importance.head(30)
plt.figure(figsize=(7,10))
plt.barh(top_30['Base_var'][::-1], top_30['Importance'][::-1], color='skyblue')
# Add labels
bars = plt.barh(top_30['Base_var'][::-1], top_30['Importance'][::-1],color='skyblue')
plt.bar_label(bars, fmt='%.3f') #3decimal place
plt.xlim(right=top_30['Importance'].max() * 1.15) #make width wider to prevent the label cross border
plt.xlabel('Aggregated Importance Score')
plt.title('Top 30 Clinical Predictors (Grouped)')
plt.show()

In [ ]:
#after checking all 30 variables,  26 features are taken (above 0.005 importance and one more for diabetes)
final_26 = ['CDRSUM', 'NACCMOCA', 'TRAILB', 'REMDATES', 'TRAILA', 'TAXES', 'TRAVEL', 
            'NACCBMI', 'BILLS', 'NACCAGE', 'PAYATTN', 'SHOPPING', 'ALCFREQ',
            'EVENTS', 'NACCGDS', 'EDUC', 'MEALPREP', 'SMOKYRS', 'MARISTAT',
            'GAMES', 'RACE', 'NACCNE4S', 'SEX', 'HYPERT', 'DIABET', 'MINTTOTS']

#extract encoded parts from each variable
final_feature = [col for col in X_train_final.columns 
                 if any(col == feature or col.startswith(feature + '_') 
                        for feature in final_26)]

#create final train and test set
X_train_top = X_train_final[final_feature]
X_test_top = X_test_final[final_feature]

print(f"Total clinical variables: {len(final_26)}")
print(f"Total encoded columns in model: {len(final_feature)}")

## b. Min-Max Scaling

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_top)
X_test_scaled = scaler.transform(X_test_top)

In [ ]:
X_train_s_df = pd.DataFrame(X_train_scaled, columns=X_train_top.columns)

#see how it look like
print("First 5 Rows of Scaled Data")
X_train_s_df.head()

# 4. SMOTETomek for train set

In [ ]:
smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train_scaled, y_train)

print(f"Final training set size after SMOTETomek: {X_train_res.shape}")
print(f"Final test set size : {X_test_scaled.shape}")

print("~ Train set Distribution~")
y_train_res.value_counts().sort_index()

# 5a. Model 1: Random Forest (RF)

## a. Train Model

In [ ]:
#rf model
#import needed library
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay, classification_report

#create rf model and fit train data in rf model
rf = RandomForestClassifier(random_state=42) #use default parameter
rf.fit(X_train_res, y_train_res)

#Get predictions
y_pred_rf = rf.predict(X_test_scaled)
y_proba_rf = rf.predict_proba(X_test_scaled)

rfacc = accuracy_score(y_test, y_pred_rf)
rf_report = classification_report(y_test, y_pred_rf, target_names=['Normal', 'MCI', 'AD Dementia'], digits=4)
rf_auc = roc_auc_score(y_test, y_proba_rf, multi_class='ovr') #One-vs-Rest for 3 class
#function to calculate specificity
def calculate_specificity(conf_matrix):
    specificity = [] #tn / (tn + fp) formula ...
    for i in range(len(conf_matrix)):
        tn = np.sum(conf_matrix) - (np.sum(conf_matrix[:, i]) + np.sum(conf_matrix[i, :]) - conf_matrix[i, i])
        fp = np.sum(conf_matrix[:, i]) - conf_matrix[i, i]
        specificity.append(tn / (tn + fp))
    return np.mean(specificity)

rf_cm = confusion_matrix(y_test, y_pred_rf)
rf_spec = calculate_specificity(rf_cm)

print("Random Forest Model performance Before Tunning:")
print(f"Accuracy:    {rfacc:.4f}")
print(f"Specificity: {rf_spec:.4f}")
print(f"AUC-ROC:     {rf_auc:.4f}")

print("\nClassification Report:\n", rf_report)

## b. Visualise the Confusion Matrix & ROC Curve

In [ ]:
#heatmap confusion metrix before prun
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'MCI', 'AD Dementia'],
            yticklabels=['Normal', 'MCI', 'AD Dementia'])

plt.title('Confusion Matrix - Random Forest Before Tunning')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

#make the output become binary for multiclass plotting
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
class_num = y_test_bin.shape[1]

colors = ['green', 'orange', 'red']
targetnames = ['Normal', 'MCI', 'AD Dementia']

for i in range(class_num):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_rf[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'ROC curve of {targetnames[i]} (area = {roc_auc:0.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) #diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (One-vs-Rest) - Random Forest Before Tunning')
plt.legend(loc="lower right")
plt.show()

## c. Hyperparameter Tuning

In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score

def rfmodel(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    }
    rfmodel = RandomForestClassifier(**params, random_state= 42)
    #use 5 cross-validation on resampled train data
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(rfmodel, X_train_res, y_train_res,
                          cv=cv, scoring='roc_auc_ovr', n_jobs=-1).mean()

#study
rfstudy = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42)) #for reprductivity
rfstudy.optimize(rfmodel, n_trials=50)

#get best parameter
print(f"Best Trial Score (AUC): {rfstudy.best_value:.4f}")
print("Best Params:", rfstudy.best_params)

In [ ]:
#fit
best_rf = RandomForestClassifier(**rfstudy.best_params,
                                 random_state=42)
best_rf.fit(X_train_res, y_train_res)

## d. Evaluate Model with Test Set

In [ ]:
#use best model to predict
y_pred_best_rf = best_rf.predict(X_test_scaled)
y_proba_best_rf = best_rf.predict_proba(X_test_scaled)

final_rfacc = accuracy_score(y_test, y_pred_best_rf)
final_rfauc = roc_auc_score(y_test, y_proba_best_rf, multi_class='ovr')
final_rfcm = confusion_matrix(y_test, y_pred_best_rf)
final_rfspec = calculate_specificity(final_rfcm)
final_rfreport = classification_report(y_test, y_pred_best_rf, target_names=['Normal', 'MCI', 'AD Dementia'], digits=4)

print("Random Forest Model Performance After Tunning:")
print(f"Final Accuracy: {final_rfacc:.4f}")
print(f"Specificity: {final_rfspec:.4f}")
print(f"Final AUC-ROC:  {final_rfauc:.4f}")
print("\nClassification Report:\n", final_rfreport)

## e. Visualise the Confusion Matrix & ROC Curve After Tuning

In [ ]:
#heatmap confusion metrix after prun
sns.heatmap(final_rfcm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'MCI', 'AD Dementia'],
            yticklabels=['Normal', 'MCI', 'AD Dementia'])

plt.title('Confusion Matrix - Random Forest After Tunning')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
#roc curve
for i in range(class_num):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_best_rf[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'ROC curve of {targetnames[i]} (area = {roc_auc:0.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) #diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (One-vs-Rest) - Random Forest After Tunning')
plt.legend(loc="lower right")
plt.show()

# 5b. Model 2: Extreme Gradient Boosting (XGBoost)

## a. Train Model

In [ ]:
#xgb model
#import needed library
import xgboost as xg

#create xgb model and fit train data in xgb model
xgb = xg.XGBClassifier(objective='multi:softprob', #multiclass
                       num_class=3,
                       eval_metric='mlogloss',
                       n_estimators=100,
                       learning_rate=0.05,
                       max_depth=6,
                       random_state=42,
                       n_jobs=-1)

xgb.fit(X_train_res, y_train_res)

#Get predictions
y_pred_xgb = xgb.predict(X_test_scaled)
y_proba_xgb = xgb.predict_proba(X_test_scaled)

xgbacc = accuracy_score(y_test, y_pred_xgb)
xgb_report = classification_report(y_test, y_pred_xgb, target_names=['Normal', 'MCI', 'AD Dementia'], digits=4)
xgb_auc = roc_auc_score(y_test, y_proba_xgb, multi_class='ovr') #One-vs-Rest for 3 class

xgb_cm = confusion_matrix(y_test, y_pred_xgb)
xgb_spec = calculate_specificity(xgb_cm)

print("XGBoost Model performance Before Tunning:")
print(f"Accuracy:    {xgbacc:.4f}")
print(f"Specificity: {xgb_spec:.4f}")
print(f"AUC-ROC:     {xgb_auc:.4f}")

print("\nClassification Report:\n", xgb_report)

## b. Visualise the Confusion Matrix & ROC Curve

In [ ]:
#heatmap confusion metrix before prun
sns.heatmap(xgb_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'MCI', 'AD Dementia'],
            yticklabels=['Normal', 'MCI', 'AD Dementia'])

plt.title('Confusion Matrix - XGBoost Before Tunning')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
#roc curve
for i in range(class_num):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_xgb[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'ROC curve of {targetnames[i]} (area = {roc_auc:0.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) #diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (One-vs-Rest) - XGBoost Before Tunning')
plt.legend(loc="lower right")
plt.show()

## c. Hyperparameter Tuning

In [ ]:
def xgbmodel(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': len(np.unique(y_train_res)),
        'random_state': 42,
        'eval_metric': 'mlogloss',
        #hyperparameter to tune
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5)
    }
    xgbmodel = xg.XGBClassifier(**params)
    #use 5 cross-validation on resampled train data
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(xgbmodel, X_train_res, y_train_res,
                          cv=cv, scoring='roc_auc_ovr', n_jobs=-1).mean()

#study
xgbstudy = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42)) #for reprductivity
xgbstudy.optimize(xgbmodel, n_trials=50)

#get best parameter
print(f"Best Trial Score (AUC): {xgbstudy.best_value:.4f}")
print("Best Params:", xgbstudy.best_params)

In [ ]:
#fit
best_xgb = xg.XGBClassifier(**xgbstudy.best_params, 
                            objective='multi:softprob',
                            num_class=len(np.unique(y_train_res)),
                            random_state=42,           
                            eval_metric= 'mlogloss')
best_xgb.fit(X_train_res, y_train_res)

## d. Evaluate Model with Test Set

In [ ]:
#use best model to predict
y_pred_best_xgb = best_xgb.predict(X_test_scaled)
y_proba_best_xgb= best_xgb.predict_proba(X_test_scaled)

final_xgbacc = accuracy_score(y_test, y_pred_best_xgb)
final_xgbauc = roc_auc_score(y_test, y_proba_best_xgb, multi_class='ovr')
final_xgbcm = confusion_matrix(y_test, y_pred_best_xgb)
final_xgbspec = calculate_specificity(final_xgbcm)
final_xgbreport = classification_report(y_test, y_pred_best_xgb, target_names=['Normal', 'MCI', 'AD Dementia'], digits=4)

print("XGBoost Model Performance After Tunning:")
print(f"Final Accuracy: {final_xgbacc:.4f}")
print(f"Specificity: {final_xgbspec:.4f}")
print(f"Final AUC-ROC:  {final_xgbauc:.4f}")
print("\nClassification Report:\n", final_xgbreport)

## e. Visualise the Confusion Matrix & ROC Curve After Tuning

In [ ]:
#heatmap confusion metrix after prun
sns.heatmap(final_xgbcm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'MCI', 'AD Dementia'],
            yticklabels=['Normal', 'MCI', 'AD Dementia'])

plt.title('Confusion Matrix - XGBoost After Tunning')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
#roc curve
for i in range(class_num):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_best_xgb[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'ROC curve of {targetnames[i]} (area = {roc_auc:0.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) #diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (One-vs-Rest) - XGBoost After Tunning')
plt.legend(loc="lower right")
plt.show()

# 5c. Model 3: Light Gradient Boosting Machine (LightGBM)

## a. Train Model

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(           
    objective='multiclass',#multiclass
    num_class=len(np.unique(y_train_res)),
    metric= "multi_logloss",           
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    n_jobs=-1
)

lgbm.fit(X_train_res, y_train_res)

#Get predictions
y_pred_lgbm = lgbm.predict(X_test_scaled)
y_proba_lgbm = lgbm.predict_proba(X_test_scaled)

lgbmacc = accuracy_score(y_test, y_pred_lgbm)
lgbm_report = classification_report(y_test, y_pred_lgbm, target_names=['Normal', 'MCI', 'AD Dementia'], digits=4)
lgbm_auc = roc_auc_score(y_test, y_proba_lgbm, multi_class='ovr') #One-vs-Rest for 3 class

lgbm_cm = confusion_matrix(y_test, y_pred_lgbm)
lgbm_spec = calculate_specificity(lgbm_cm)

print("LightGBM Model performance Before Tunning:")
print(f"Accuracy:    {lgbmacc:.4f}")
print(f"Specificity: {lgbm_spec:.4f}")
print(f"AUC-ROC:     {lgbm_auc:.4f}")

print("\nClassification Report:\n", lgbm_report)

## b. Visualise the Confusion Matrix & ROC Curve

In [ ]:
#heatmap confusion metrix before prun
sns.heatmap(lgbm_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'MCI', 'AD Dementia'],
            yticklabels=['Normal', 'MCI', 'AD Dementia'])

plt.title('Confusion Matrix - LightGBM Before Tunning')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
#roc curve
for i in range(class_num):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_lgbm[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'ROC curve of {targetnames[i]} (area = {roc_auc:0.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) #diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (One-vs-Rest) - LightGBM Before Tunning')
plt.legend(loc="lower right")
plt.show()

## c. Hyperparameter Tuning

In [ ]:
def lgbmmodel(trial):
    params = {
        'objective': 'multiclass',
        'num_class':len(np.unique(y_train_res)),
        'random_state': 42,
        "metric": "multi_logloss",
        "n_jobs": -1,
        #hyperparameter to tune
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50)
    }
    lgbmmodel = LGBMClassifier(**params)
    #use 5 cross-validation on resampled train data
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(lgbmmodel, X_train_res, y_train_res,
                          cv=cv, scoring='roc_auc_ovr', n_jobs=-1).mean()

#study
lgbmstudy = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42)) #for reprductivity
lgbmstudy.optimize(lgbmmodel, n_trials=50)

#get best parameter
print(f"Best Trial Score (AUC): {lgbmstudy.best_value:.4f}")
print("Best Params:", lgbmstudy.best_params)

In [ ]:
#fit
best_lgbm = LGBMClassifier(**lgbmstudy.best_params, 
                            objective='multiclass',
                            num_class=len(np.unique(y_train_res)),
                            random_state=42,           
                            metric= "multi_logloss")
best_lgbm.fit(X_train_res, y_train_res)

## d. Evaluate Model with Test Set

In [ ]:
#use best model to predict
y_pred_best_lgbm = best_lgbm.predict(X_test_scaled)
y_proba_best_lgbm= best_lgbm.predict_proba(X_test_scaled)

final_lgbmacc = accuracy_score(y_test, y_pred_best_lgbm)
final_lgbmauc = roc_auc_score(y_test, y_proba_best_lgbm, multi_class='ovr')
final_lgbmcm = confusion_matrix(y_test, y_pred_best_lgbm)
final_lgbmspec = calculate_specificity(final_lgbmcm)
final_lgbmreport = classification_report(y_test, y_pred_best_lgbm, target_names=['Normal', 'MCI', 'AD Dementia'], digits=4)

print("LightGBM Model Performance After Tunning:")
print(f"Final Accuracy: {final_lgbmacc:.4f}")
print(f"Specificity: {final_lgbmspec:.4f}")
print(f"Final AUC-ROC:  {final_lgbmauc:.4f}")
print("\nClassification Report:\n", final_lgbmreport)

## e. Visualise the Confusion Matrix & ROC Curve After Tuning

In [ ]:
#heatmap confusion metrix before prun
sns.heatmap(final_lgbmcm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'MCI', 'AD Dementia'],
            yticklabels=['Normal', 'MCI', 'AD Dementia'])

plt.title('Confusion Matrix - LightGBM After Tunning')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
#roc curve
for i in range(class_num):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_best_lgbm[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'ROC curve of {targetnames[i]} (area = {roc_auc:0.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) #diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (One-vs-Rest) - LightGBM After Tunning')
plt.legend(loc="lower right")
plt.show()

# 5d. Model 4: Support Vector Machine (SVM)

## a. Train Model

In [ ]:
from sklearn.svm import SVC

svm = SVC(
    kernel='rbf',   
    C=1.0, # regularissation, default          
    gamma='scale',     #kernel coefficient
    probability=True,  #need for SHAP and AUC
    random_state=42
)

svm.fit(X_train_res, y_train_res)

#Get predictions
y_pred_svm = svm.predict(X_test_scaled)
y_proba_svm = svm.predict_proba(X_test_scaled)

svmacc = accuracy_score(y_test, y_pred_svm)
svm_report = classification_report(y_test, y_pred_svm, target_names=['Normal', 'MCI', 'AD Dementia'], digits=4)
svm_auc = roc_auc_score(y_test, y_proba_svm, multi_class='ovr') #One-vs-Rest for 3 class

svm_cm = confusion_matrix(y_test, y_pred_svm)
svm_spec = calculate_specificity(svm_cm)

print("SVM Model performance Before Tunning:")
print(f"Accuracy:    {svmacc:.4f}")
print(f"Specificity: {svm_spec:.4f}")
print(f"AUC-ROC:     {svm_auc:.4f}")

print("\nClassification Report:\n", svm_report)

## b. Visualise the Confusion Matrix & ROC Curve

In [ ]:
#heatmap confusion metrix before prun
sns.heatmap(svm_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'MCI', 'AD Dementia'],
            yticklabels=['Normal', 'MCI', 'AD Dementia'])

plt.title('Confusion Matrix - SVM Before Tunning')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
#roc curve
for i in range(class_num):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_svm[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'ROC curve of {targetnames[i]} (area = {roc_auc:0.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) #diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (One-vs-Rest) - SVM Before Tunning')
plt.legend(loc="lower right")
plt.show()

## c. Hyperparameter Tuning

In [ ]:
def svmmodel(trial):
    params = {
        'C': trial.suggest_float('C', 0.1, 100, log=True),
        'gamma': trial.suggest_float('gamma', 1e-4, 1, log=True),
        'kernel': trial.suggest_categorical('kernel', ['rbf', 'sigmoid']),
        'probability': True,
        'random_state': 42
    }
    svmmodel = SVC(**params)
    #use 5 cross-validation on resampled train data
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(svmmodel, X_train_res, y_train_res,
                          cv=cv, scoring='roc_auc_ovr', n_jobs=-1).mean()

#study
svmstudy = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42)) #for reprductivity
svmstudy.optimize(svmmodel, n_trials=50)

#get best parameter
print(f"Best Trial Score (AUC): {svmstudy.best_value:.4f}")
print("Best Params:", svmstudy.best_params)

In [ ]:
#fit
best_svm = SVC(**svmstudy.best_params, 
                probability=True,       
                random_state=42)
best_svm.fit(X_train_res, y_train_res)

## d. Evaluate Model with Test Set

In [ ]:
#use best model to predict
y_pred_best_svm = best_svm.predict(X_test_scaled)
y_proba_best_svm= best_svm.predict_proba(X_test_scaled)

final_svmacc = accuracy_score(y_test, y_pred_best_svm)
final_svmauc = roc_auc_score(y_test, y_proba_best_svm, multi_class='ovr')
final_svmcm = confusion_matrix(y_test, y_pred_best_svm)
final_svmspec = calculate_specificity(final_svmcm)
final_svmreport = classification_report(y_test, y_pred_best_svm, target_names=['Normal', 'MCI', 'AD Dementia'], digits=4)

print("SVM Model Performance After Tunning:")
print(f"Final Accuracy: {final_svmacc:.4f}")
print(f"Specificity: {final_svmspec:.4f}")
print(f"Final AUC-ROC:  {final_svmauc:.4f}")
print("\nClassification Report:\n", final_svmreport)

## e. Visualise the Confusion Matrix & ROC Curve After Tuning

In [ ]:
#heatmap confusion metrix after prun
sns.heatmap(final_svmcm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'MCI', 'AD Dementia'],
            yticklabels=['Normal', 'MCI', 'AD Dementia'])

plt.title('Confusion Matrix - SVM After Tunning')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
#roc curve
for i in range(class_num):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_best_svm[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'ROC curve of {targetnames[i]} (area = {roc_auc:0.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) #diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (One-vs-Rest) - SVM After Tunning')
plt.legend(loc="lower right")
plt.show()

# 6. SHAP analysis for Random Forest

In [ ]:
#convert to dataframe so shap can show feature name
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=final_feature, 
    index=X_test_top.index
)

In [ ]:
import shap

#rf is the most best performing model
rf_explainer = shap.TreeExplainer(best_rf)

# Calculate SHAP values 
shap_values = rf_explainer.shap_values(X_test_scaled_df)
shap_values_MCI = shap_values[:, :, 1] #for mci case
shap_values_AD  = shap_values[:, :, 2] #for ad dementia case


In [ ]:
# shap_values- a list of arrays for multiclass(one array per class)
#Summary plot for MCI classes (show top 30)
shap.summary_plot(shap_values_MCI, X_test_scaled_df, max_display=30, show=False)
plt.title("SHAP Summary Plot for MCI Prediction", fontsize=14, fontweight='bold')
plt.show()

#Summary plot for AD dementia classes (show top 30)
shap.summary_plot(shap_values_AD, X_test_scaled_df, max_display=30, show=False)
plt.title("SHAP Summary Plot for AD Dementia Prediction", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
#Bar plot for Global feature importance
#take absolute mean (remove negative sign)
global_shap_values = np.mean(np.abs(shap_values), axis=2)

#(show top 30)
shap.summary_plot(global_shap_values, X_test_scaled_df, plot_type="bar", max_display=30, show=False)
plt.title("Average Global Feature Importance", fontsize=14, fontweight='bold')
plt.xlabel("Mean Absolute SHAP Value (Average Impact on Model Output)")
plt.show()

# 7. Save chosen RF model and other needed files for system

In [ ]:
#use joblib to save it as a file
#will be used in external validation using ADNI dataset
import joblib

#save scaler used
joblib.dump(scaler, 'minmax_scaler.pkl')
#save encoder
joblib.dump(ohe, 'onehot_encoder.pkl')
#save chosen encoded features list
joblib.dump(final_feature, 'selected_features.pkl')
#save trained model
joblib.dump(best_rf, 'best_rf_model.pkl')

#just in case, need to record the imputation values
#when doctor didnt fill value for some feature, it is needed
con_var = [v for v in con_var if v in nacc_cleaned.columns]
cat_var = [v for v in cat_var if v in nacc_cleaned.columns]
imputer_values = {
    "con_median": nacc_cleaned[con_var].median().to_dict(),
    "cat_mode": nacc_cleaned[cat_var].mode().iloc[0].to_dict()
}

joblib.dump(imputer_values, "imputer_value.pkl")


# **Extract A subset of NACC dataset for system testing

In [ ]:
import sqlite3
chosen_features = ['NACCID', 'NACCALZD', 'NACCUDSD', 'CDRSUM', 'NACCMOCA', 'TRAILB', 'REMDATES', 'TRAILA', 'TAXES', 'TRAVEL', 
                    'NACCBMI', 'BILLS', 'NACCAGE', 'PAYATTN', 'SHOPPING', 'ALCFREQ',
                    'EVENTS', 'NACCGDS', 'EDUC', 'MEALPREP', 'SMOKYRS', 'MARISTAT',
                    'GAMES', 'RACE', 'NACCNE4S', 'SEX', 'HYPERT', 'DIABET', 'MINTTOTS']
nacc_subset_raw = nacc_baseline[chosen_features].copy()
print(f"Initial baseline rows: {len(nacc_subset_raw)}")

#again do the step
#-------------------------------------------------------------
nacc_subset_modified0 = nacc_subset_raw.replace(-4, np.nan).copy()
#Regroup the target variable NACCUDSD
nacc_subset_modified1 = nacc_subset_modified0[nacc_subset_modified0['NACCUDSD'] != 2].copy()

#filter dementia, only keep Alzheimer's Disease Dementia (project focus)
#use NACCALZD to keep AD dementia from NACCUDSD
three_classes = (nacc_subset_modified1['NACCUDSD'].isin([1, 3])) | ((nacc_subset_modified1['NACCUDSD'] == 4) & (nacc_subset_modified1['NACCALZD'] == 1))
nacc_subset_modified1 = nacc_subset_modified1[three_classes].copy()

#Drop the NACCALZD column
nacc_subset_modified2 = nacc_subset_modified1.drop(columns=['NACCALZD'])

#-------------------------------------------------------------
#trail a and b have problem code 995-998, change to nan
if 'TRAILB' in nacc_subset_modified2.columns:
    nacc_subset_modified2['TRAILB'] = nacc_subset_modified2['TRAILB'].replace([995, 996, 997, 998], np.nan)

if 'TRAILA' in nacc_subset_modified2.columns:
    nacc_subset_modified2['TRAILA'] = nacc_subset_modified2['TRAILA'].replace([995, 996, 997, 998], np.nan)

#mint has 95-98 problem code, change to nan
if 'MINTTOTS' in nacc_subset_modified2.columns:
    nacc_subset_modified2['MINTTOTS'] = nacc_subset_modified2['MINTTOTS'].replace([95, 96, 97, 98], np.nan)


#variables that need careful clean because score range overlap with missing code
critical_var = ['NACCMOCA', 'NACCGDS', 'CDRSUM', 'EDUC', 'NACCUDSD', 'SMOKYRS']
critical_exist = [c for c in critical_var if c in nacc_subset_modified2.columns]
trail_var = ['TRAILA', 'TRAILB']
problem_s_var = ['MINTTOTS']

#other category col (category feature normally dont have more than 10 categories)
other_cols = set(trail_var + problem_s_var + critical_exist + ['NACCAGE'])
general_cols = [c for c in nacc_subset_modified2.columns if c not in other_cols]
cat_cols = [col for col in general_cols 
            if nacc_subset_modified2[col].nunique(dropna=True) <= 10]


#clean
nacc_subset_modified2[general_cols] = nacc_subset_modified2[general_cols].replace(trash_code_b, np.nan)
nacc_subset_modified2[critical_exist] = nacc_subset_modified2[critical_exist].replace(trash_code_b, np.nan)
nacc_subset_modified2[cat_cols] = nacc_subset_modified2[cat_cols].replace(trash_code_s, np.nan)

#check total missing values
print(f"Total Missing Values: {nacc_subset_modified2.isnull().sum().sum()}")

#-------------------------------------------------------------
#since important variable NACCMoCA has more than 50% missing, cannot impute
#only can remove missing rows
nacc_subset_modified3 = nacc_subset_modified2.dropna(subset=['NACCMOCA']).copy()

print(f"Rows remaining after MoCA filtering: {nacc_subset_modified3.shape[0]}")
print(f"Total Missing Values Left: {nacc_subset_modified3.isnull().sum().sum()}")

#-------------------------------------------------------------
target_map = {1: 0, 3: 1, 4: 2}
nacc_subset_modified3['STATUS'] = nacc_subset_modified3['NACCUDSD'].map(target_map)
#drop original NACCUDSD column
nacc_subset_modified3 = nacc_subset_modified3.drop(columns=['NACCUDSD'])

#-------------------------------------------------------------
# Extracting the 1000 record demo subset...
df_1000 = nacc_subset_modified3.sample(n=1000, random_state=42).copy()

#-------------------------------------------------------------
#Compiling clean database file...
conn = sqlite3.connect("cogneutest.db")
df_1000.to_sql("patient_records", conn, if_exists="replace", index=False)
conn.close()
print("\n Done!")